<a href="https://colab.research.google.com/github/navap3206-debug/hello-world/blob/main/C2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Laboratorio de regresión logística

|                |   |
:----------------|---|
| **Nombre**     |Carlos Nava   |
| **Fecha**      |16/03/2026  |
| **Expediente** |756213   |

In machine learning, Support Vector Machines (SVM) are supervised learning models with associated learning algorithms that analyze data used for classification and regression analysis. It is mostly used in classification problems. In this algorithm, each data item is plotted as a point in p-dimensional space (where p is the number of features), with the value of each feature being the value of a particular coordinate. Then, classification is performed by finding the hyper-plane that best differentiates the two classes (or more if we have a multi class problem):

$$ f(x) = w^T \varphi(x) + b $$

where $\varphi: X \rightarrow F $ is a function that makes each input point $x$ correspond to a point in F, where F is a Hilbert space.

In addition to performing linear classification, SVMs can efficiently perform a non-linear classification, implicitly mapping their inputs into high-dimensional feature spaces (more specifically using the kernel trick, like the RBF funcion).

[1]

OLS utilizes the squared residuals to fit the parameters. Large residuals caused by outliers may worsen the accuracy significantly.

Support Vectors use piecewise linear functions to counter this, in which a hyperparameter  $\epsilon$ called the margin lets errors that are less or equal to it be 0, and error larger than it be $e - \epsilon$.

The problem to solve is:

\begin{split}
        \min_{w, b, \xi, \xi^*} \mathcal{P}_\epsilon(w, b, \xi) &= \frac{1}{2} w^T w + c \sum_{k=1}^{N} \xi_k \\
        \text{s.t. } & y_k [w^T \varphi(x_k) - b] \geq 1- \xi_k,\ \ k = 1, ..., N \\
        & \xi_k \geq 0,\ \ k = 1, ..., N
\end{split}


The most important question that arises when using a SVM is how to choose the correct hyperplane. Consider the following scenarios:

### Scenario 1

In this scenario there are three hyperplanes called A, B, and C. Now, the problem is to identify the hyperplane which best differentiates the stars and the circles.

<center><img src="https://media.geeksforgeeks.org/wp-content/uploads/SVM_21-2.png" alt="what image shows"></center>

In this case, hyperplane B separates the stars and the circle betters, hence it is the correct hyperplane.


### Scenario 2

Now take another scenario where all three hyperplanes are segregating classes well. The question that arises is how to choose the best hyperplane in this situation.

<center><img src="https://media.geeksforgeeks.org/wp-content/uploads/SVM_4-2.png" alt="what image shows"></center>

In such scenarios, we calculate the margin (which is the distance between nearest data point and the hyperplane). The hyperplane with the largest margin will be considered as the correct hyperplane to classify the dataset.

Here C has the largest margin. Hence, it is considered as the best hyperplane.


### Kernels
Knowing
$$ w = \sum_{k=1}^{N} \alpha_k y_k \varphi(x_k) $$

And
$$ y_{pred} = w^T \varphi(x) + b $$

Then
$$ y_{pred} = (\sum_{k=1}^{N} \alpha_k y_k \varphi(x_k))^T \varphi(x) + b $$

Where $\varphi$ is a function that makes each input in $x$ correspond to a point in $F$ (a Hilbert space). This can be seen as processing and transforming the input featuers to keep the model's convexity. [2]

This also allows us to transform the inputs into another space where they might be more easily classified.

<center><img src=https://miro.medium.com/max/838/1*gXvhD4IomaC9Jb37tzDUVg.png alt="what image shows"></center>

## ROC and AUC

A ROC (Receiver Operating Characteristic) is a graph that shows how the classification model performs at the classification thresholds.

ROC curves typically feature true positive rate on the Y axis, and false positive rate on the X axis. This means that the top left corner of the plot is the “ideal” point - a false positive rate of zero, and a true positive rate of one. This is not very realistic, but it does mean that a larger area under the curve (AUC) is usually better. [3]

True Positive Rate is a synonym for Recall and defined as:
$$ TPR = \frac{TP}{TP + FN} $$

False Positive Rate is a synonym for Specificity and defined as:

$$ FPR = \frac{FP}{FP + TN} $$

ROC curves are typically used in binary classification to study the output of a classifier. In order to extend ROC curve and ROC area to multi-label classification, it is necessary to binarize the output. One ROC curve can be drawn per label, but one can also draw a ROC curve by considering each element of the label indicator matrix as a binary prediction (micro-averaging).

E.g. If you lower a classification threshold, more items would be classified as positive, increasing False Positives and True Positives.

AUC stands for Area under the ROC.

## Ejercicio 1

- Utiliza el dataset `Iris`, modela con SVC y haz Cross-Validation de diferentes kernels ('linear', 'poly', 'rbf', 'sigmoid').
- Modela con LogisticRegression.
- El método de Cross-Validation es K-Folds con $k=10$.
- Utiliza el AUC como métrico de Cross-Validation.
- Compara resultados.

In [ ]:
import numpy as np
from google.colab import files
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import train_test_split


Importamos el datase

In [ ]:
from sklearn.datasets import load_iris
data = load_iris()


In [ ]:
#transformamos el dataset en un datafame
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name='species')
iris_df = pd.concat([X, y], axis=1)
iris_df['species'] = iris_df['species'].map(lambda x: data.target_names[x])
iris_df.info()
iris_df.head(5)
iris_df['species'].value_counts()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   sepal length (cm)  150 non-null    float64
 1   sepal width (cm)   150 non-null    float64
 2   petal length (cm)  150 non-null    float64
 3   petal width (cm)   150 non-null    float64
 4   species            150 non-null    object 
dtypes: float64(4), object(1)
memory usage: 6.0+ KB


,count
species,
setosa,50
versicolor,50
virginica,50


In [ ]:
#Purgamos la tercer categoria para trabajar unicamente con dos.
iris_df_log = iris_df[iris_df['species'].isin(['setosa', 'versicolor'])].copy()
iris_df_log['species'] = iris_df_log['species'].map({'setosa': 0, 'versicolor': 1})

X_binary = iris_df_log.drop('species', axis=1)
y_binary = iris_df_log['species']

y_binary.value_counts()

,count
species,
0,50
1,50


In [ ]:
iris_df_log.head(5)


,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),species
0,5.1,3.5,1.4,0.2,0
1,4.9,3.0,1.4,0.2,0
2,4.7,3.2,1.3,0.2,0
3,4.6,3.1,1.5,0.2,0
4,5.0,3.6,1.4,0.2,0


Identificamos las columnas a escalar y cual es nuestra variable a predecir

In [ ]:
numerical_columns = ['sepal length (cm)','sepal width (cm)',	'petal length (cm)',	'petal width (cm)']
categorical_columns = []
target_col = ["species"]
X = iris_df_log[numerical_columns + categorical_columns]
y = iris_df_log[target_col]

Establecemos los modelos mediante SVC y los 4 kernels a entrenar

In [ ]:
#('linear', 'poly', 'rbf', 'sigmoid')
num_transformer = StandardScaler()
cat_transformer = OneHotEncoder(drop='first')
model = SVC(probability = True, kernel = 'linear')
model_2  = SVC(probability = True, kernel = 'poly')
model_3  = SVC(probability = True, kernel = 'rbf')
model_4  = SVC(probability = True, kernel = 'sigmoid')

Aplicamosel preprocesamiento y aplicamos el pipeline con los elementos ya declarados

In [ ]:
preprocesor = ColumnTransformer(transformers=[('num',num_transformer,numerical_columns), ('cat', cat_transformer, categorical_columns)])
pipeline_1 = Pipeline(steps=[('preprocesor', preprocesor), ('model', model)])
pipeline_2 = Pipeline(steps=[('preprocesor', preprocesor), ('model', model_2)])
pipeline_3 = Pipeline(steps=[('preprocesor', preprocesor), ('model', model_3)])
pipeline_4 = Pipeline(steps=[('preprocesor', preprocesor), ('model', model_4)])

In [ ]:
#ENTRENAMOS
pipeline_1.fit(X, y)
pipeline_2.fit(X, y)
pipeline_3.fit(X, y)
pipeline_4.fit(X,y)

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for exam

Pipeline(steps=[('preprocesor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['sepal length (cm)',
                                                   'sepal width (cm)',
                                                   'petal length (cm)',
                                                   'petal width (cm)']),
                                                 ('cat',
                                                  OneHotEncoder(drop='first'),
                                                  [])])),
                ('model', SVC(kernel='sigmoid', probability=True))])

APLICAMOS CROSS VALIDATION MEDINATE K FOLDS DIVIDIENDO EN 10 SETS DE TEST-TRAIN

In [ ]:
#RECOPILAMOS LOS SCORES DE K FOLDS
scores1 = pd.DataFrame(cross_val_score(pipeline_1, X, y, cv=10, scoring='accuracy'), columns=['accuracy'])
scores2 = pd.DataFrame(cross_val_score(pipeline_2, X, y, cv=10, scoring='accuracy'), columns=['accuracy'])
scores3 = pd.DataFrame(cross_val_score(pipeline_3, X, y, cv=10, scoring='accuracy'), columns=['accuracy'])
scores4 = pd.DataFrame(cross_val_score(pipeline_4, X, y, cv=10, scoring='accuracy'), columns=['accuracy'])
scores = pd.concat([scores1, scores2, scores3, scores4], axis=1)
scores


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for exam

,accuracy,accuracy,accuracy,accuracy
0,1.0,1.0,1.0,1.0
1,1.0,1.0,1.0,1.0
2,1.0,1.0,1.0,1.0
3,1.0,1.0,1.0,1.0
4,1.0,1.0,1.0,1.0
5,1.0,1.0,1.0,1.0
6,1.0,1.0,1.0,1.0
7,1.0,1.0,1.0,1.0
8,1.0,1.0,1.0,1.0
9,1.0,1.0,1.0,1.0


In [ ]:
scores.describe()

,accuracy,accuracy,accuracy,accuracy
count,10.0,10.0,10.0,10.0
mean,1.0,1.0,1.0,1.0
std,0.0,0.0,0.0,0.0
min,1.0,1.0,1.0,1.0
25%,1.0,1.0,1.0,1.0
50%,1.0,1.0,1.0,1.0
75%,1.0,1.0,1.0,1.0
max,1.0,1.0,1.0,1.0


Parece ser que estas variables tinene una precision predictoria perfecta.

## Ejercicio 2
- Repite el ejercicio 1 con el dataset `Default`. Utiliza `default` como target.

In [14]:
#importamos el dataset
df = files.upload()

Saving Default.csv to Default.csv


In [15]:
default = pd.read_csv('Default.csv')
default.head(5)

,default,student,balance,income
0,No,No,729.526495,44361.625074
1,No,Yes,817.180407,12106.134700
2,No,No,1073.549164,31767.138950
3,No,No,529.250605,35704.493940
4,No,No,785.655883,38463.495880


In [16]:
#tranformo a forma binaria
default['default'] = default['default'].map({'No': 0, 'Yes':1})
default['student'] = default['student'].map({'No': 0, 'Yes':1})
default.head(5)

,default,student,balance,income
0,0,0,729.526495,44361.625074
1,0,1,817.180407,12106.134700
2,0,0,1073.549164,31767.138950
3,0,0,529.250605,35704.493940
4,0,0,785.655883,38463.495880


ESTABLECEMOS LAS COLUMNAS PARA EL PIPELINE

In [22]:
numerical_columns = ["balance", "income"]
categorical_columns = ["student"]
target_col = ["default"]
X_2 = default[numerical_columns + categorical_columns]
y_2 = default[target_col]

In [27]:
#LOS MODELOS YA ESTÁN DECLARADOS SOLO BATA CON ESTBALCER NUEVOS PIPELINES CON LOS DATOS NUEVOS
preprocesor_2 = ColumnTransformer(transformers=[('num',num_transformer,numerical_columns), ('cat', cat_transformer, categorical_columns)])
pipeline_1 = Pipeline(steps=[('preprocesor', preprocesor_2), ('model', model)])
pipeline_2 = Pipeline(steps=[('preprocesor', preprocesor_2), ('model', model_2)])
pipeline_3 = Pipeline(steps=[('preprocesor', preprocesor_2), ('model', model_3)])
pipeline_4 = Pipeline(steps=[('preprocesor', preprocesor_2), ('model', model_4)])

In [28]:
#ENTRENAMIENTO CON LOS 4 KERNELS
pipeline_1.fit(X_2,y_2)
pipeline_2.fit(X_2,y_2)
pipeline_3.fit(X_2,y_2)
pipeline_4.fit(X_2,y_2)


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for exam

Pipeline(steps=[('preprocesor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['balance', 'income']),
                                                 ('cat',
                                                  OneHotEncoder(drop='first'),
                                                  ['student'])])),
                ('model', SVC(kernel='sigmoid', probability=True))])

In [30]:
#RECOPILO LOS RESULTADOS DE CADA CROSS VALIDATION
scores1 = pd.DataFrame(cross_val_score(pipeline_1, X_2, y_2, cv=10, scoring='roc_auc'), columns=['roc_auc'])
scores2 = pd.DataFrame(cross_val_score(pipeline_2, X_2, y_2, cv=10, scoring='roc_auc'), columns=['roc_auc'])
scores3 = pd.DataFrame(cross_val_score(pipeline_3, X_2, y_2, cv=10, scoring='roc_auc'), columns=['roc_auc'])
scores4 = pd.DataFrame(cross_val_score(pipeline_4, X_2, y_2, cv=10, scoring='roc_auc'), columns=['roc_auc'])
scores = pd.concat([scores1, scores2, scores3, scores4], axis=1)
scores

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for exam

,roc_auc,roc_auc,roc_auc,roc_auc
0,0.921845,0.885463,0.733415,0.783116
1,0.917270,0.893172,0.731597,0.715615
2,0.958290,0.916455,0.810410,0.875748
3,0.936135,0.895052,0.818652,0.846197
4,0.942590,0.877660,0.753847,0.806963
5,0.935633,0.883676,0.817116,0.822287
6,0.964652,0.927674,0.760584,0.774216
7,0.939867,0.897820,0.771374,0.788120
8,0.856108,0.887864,0.805596,0.811960
9,0.917793,0.862502,0.768999,0.757764


In [31]:
scores.describe()

,roc_auc,roc_auc,roc_auc,roc_auc
count,10.000000,10.000000,10.000000,10.000000
mean,0.929018,0.892734,0.777159,0.798199
std,0.030100,0.018603,0.033566,0.045367
min,0.856108,0.862502,0.731597,0.715615
25%,0.918806,0.884123,0.755531,0.776441
50%,0.935884,0.890518,0.770186,0.797541
75%,0.941910,0.897128,0.809207,0.819705
max,0.964652,0.927674,0.818652,0.875748


PARECE SER QUE EL PRIMER (linear) KERNEL POSEE MEJOR PUNTUACION DE AUC (acurrancy) QUE EL RESTO, aunque con una desviación similar al resto


# Addendum

Métricos disponibles para clasificación:
- ‘accuracy’
- ‘balanced_accuracy’
- ‘top_k_accuracy’
- ‘average_precision’
- ‘neg_brier_score’
- ‘f1’
- ‘f1_micro’
- ‘f1_macro’
- ‘f1_weighted’
- ‘f1_samples’
- ‘neg_log_loss’
- ‘precision’ etc.
- ‘recall’ etc.
- ‘jaccard’ etc.
- ‘roc_auc’
- ‘roc_auc_ovr’
- ‘roc_auc_ovo’
- ‘roc_auc_ovr_weighted’
- ‘roc_auc_ovo_weighted’
- ‘d2_log_loss_score’

# References

[1] Shigeo Abe.Support Vector Machines for Pattern Classification,2Ed.Springer-Verlag London,2010. ISBN978-1-84996-097-7. URLhttps://www.springer.com/gp/book/9781849960977.

[2] Johan A K Suykens, Tony Van Gestel, Jos De Brabanter, BartDe Moor, and Joos Vandewalle.Least Squares Support VectorMachines. World Scientific,2002. ISBN9789812381514. URLhttps://www.worldscientific.com/worldscibooks/10.1142/5089.

[3] Bradley, A. P. (1997). The use of the area under the ROC curve in the evaluation of machine learning algorithms. Pattern recognition, 30(7), 1145-1159. URL https://www.researchgate.net/post/how_can_I_interpret_the_ROC_curve_result